# Notebook 2: Análisis de rendimiento mediante QKDMeter
## Calcula QBER, % sifted bit y SKR a partir del dataset CSV

**Entrada:** CSV generados por el Notebook 1.  
**Salida:** CSV de métricas con media y desviación estadística por (escenario, distancia).

### Métricas calculadas

**QBER:**
$$\text{QBER} = \frac{\#\{\text{bit\_alice} \neq \text{bit\_bob}\}}{n_{\text{sifted}}}$$

Bits detectados tras sifting:
$$\text{P\_sift} = \frac{n_{\text{sifted}}}{N_{\text{pulsos}}}$$

**SKR** usando información mutua empírica:
$$I(A:B) = \sum_{a,b} P(a,b)\log_2\frac{P(a,b)}{P(a)P(b)}, \quad I(A:E) = \sum_{a,e} P(a,e)\log_2\frac{P(a,e)}{P(a)P(e)}$$
$$\text{SKR} = f_{\text{pulsos}}\cdot P_{\text{sift}}[I(A:B) - I(A:E)]$$


## 0. Imports

In [2]:
import numpy as np
import pandas as pd
import os, glob, json
from itertools import product
from pathlib import Path

## 1. Configuración

In [3]:
CSV_DIR    = "output_csv"   # Directorio con los CSV del Notebook 1
METRICS_DIR = "output_metrics"
os.makedirs(METRICS_DIR, exist_ok=True)

N_PULSES = 100_000           # Debe coincidir con el Notebook 1


## 2. Funciones de métricas

In [4]:
def calc_qber(df_seed_dist: pd.DataFrame) -> float:
    """
    QBER para un subconjunto (semilla, distancia).
    Retorna NaN si no hay bits sifted.
    """
    n = len(df_seed_dist)
    if n == 0:
        return np.nan
    errors = (df_seed_dist["bit_alice"] != df_seed_dist["bit_bob"]).sum()
    return float(errors / n)


def P_sifted(df_seed_dist: pd.DataFrame) -> float:
    """
    Porcentaje de bits detectados = n_sifted / N_PULSES.
    """
    return float(len(df_seed_dist) / N_PULSES)


def mutual_info(x: np.ndarray, y: np.ndarray) -> float:
    """
    Información mutua empírica I(X;Y) estimada a partir de frecuencias.
    x, y: arrays de enteros {0,1}.
    """
    n = len(x)
    if n == 0:
        return 0.0
    vals = [0, 1]
    mi = 0.0
    for a, b in product(vals, vals):
        p_ab = np.mean((x == a) & (y == b))
        p_a  = np.mean(x == a)
        p_b  = np.mean(y == b)
        if p_ab > 0 and p_a > 0 and p_b > 0:
            mi += p_ab * np.log2(p_ab / (p_a * p_b))
    return max(0.0, mi)


def calc_skr(df_seed_dist: pd.DataFrame) -> float:
    """
    SKR = I(A:B) - I(A:E) por bit sifted.
    Si Eve no interceptó ningún bit (bit_eve todo NaN), I(A:E)=0.
    Retorna 0 si SKR < 0.
    """
    if len(df_seed_dist) == 0:
        return 0.0
    a = df_seed_dist["bit_alice"].values.astype(int)
    b = df_seed_dist["bit_bob"].values.astype(int)
    iab = mutual_info(a, b)

    # Solo bits donde Eve actuó
    eve_rows = df_seed_dist["bit_eve"].notna()
    if eve_rows.sum() > 0:
        a_e = df_seed_dist.loc[eve_rows, "bit_alice"].values.astype(int)
        e   = df_seed_dist.loc[eve_rows, "bit_eve"].values.astype(int)
        iae = mutual_info(a_e, e)
    else:
        iae = 0.0

    SKF= max(0.0, iab - iae)   
    f_pulsos=1 # Para calcular el SKR por pulso emitido por Alice
    p_sifted=P_sifted(df_seed_dist)

    SKR=f_pulsos*p_sifted*SKF
    
    return SKR


## 3. Función principal: calcular métricas para un CSV

In [5]:
def compute_metrics(csv_path: str,
                    seeds: list | None = None,
                    distances: list | None = None) -> pd.DataFrame:
    """
    Lee el CSV de simulación y calcula QBER, absorción y SKR para
    cada combinación (distancia, semilla), luego agrega media y std.

    Parámetros
    ----------
    csv_path  : ruta al CSV de simulación.
    seeds     : lista de semillas a considerar (None = todas).
    distances : lista de distancias a considerar (None = todas).

    Retorna
    -------
    DataFrame con columnas:
        distancia,
        qber_mean, qber_std,
        absorption_mean, absorption_std,
        skr_mean, skr_std
    """
    df = pd.read_csv(csv_path)

    if seeds is not None:
        df = df[df["experimento"].isin(seeds)]
    if distances is not None:
        df = df[df["distancia"].isin(distances)]

    all_dists = sorted(df["distancia"].unique())
    all_seeds = sorted(df["experimento"].unique())

    rows = []
    for dist in all_dists:
        qbers, absorbs, skrs = [], [], []
        for seed in all_seeds:
            sub = df[(df["distancia"] == dist) & (df["experimento"] == seed)]
            qbers.append(calc_qber(sub))
            absorbs.append(P_sifted(sub))
            skrs.append(calc_skr(sub))

        qbers   = np.array(qbers,   dtype=float)
        absorbs = np.array(absorbs, dtype=float)
        skrs    = np.array(skrs,    dtype=float)

        row = {
            "distancia"       : dist,
            "qber_mean"       : np.nanmean(qbers),
            "qber_std"        : np.nanstd(qbers)  if len(all_seeds) > 1 else np.nan,
            "absorption_mean" : np.nanmean(absorbs),
            "absorption_std"  : np.nanstd(absorbs) if len(all_seeds) > 1 else np.nan,
            "skr_mean"        : np.nanmean(skrs),
            "skr_std"         : np.nanstd(skrs)   if len(all_seeds) > 1 else np.nan,
        }
        # Guardar también valores por semilla
        for i, seed in enumerate(all_seeds):
            row[f"qber_seed_{seed}"]       = qbers[i]
            row[f"absorption_seed_{seed}"] = absorbs[i]
            row[f"skr_seed_{seed}"]        = skrs[i]
        rows.append(row)

    return pd.DataFrame(rows)


## 4. Calcular métricas para todos los escenarios

In [6]:
csv_files = sorted(glob.glob(os.path.join(CSV_DIR, "scenario_*.csv")))
if not csv_files:
    raise FileNotFoundError(f"No se encontraron CSV en '{CSV_DIR}'. "
                            "Ejecuta primero el Notebook 1.")

print(f"CSV encontrados: {len(csv_files)}\n")

# Puedes filtrar semillas y distancias aquí si quieres un subconjunto:
SEEDS_FILTER     = None   # None = todas;  ej: [42, 123]
DISTANCES_FILTER = None   # None = todas;  ej: [0, 10, 20]

metrics_paths = {}
for csv_path in csv_files:
    sc_name = Path(csv_path).stem  # e.g. scenario_01_ideal
    print(f"  Procesando: {sc_name} …", end=" ")

    df_metrics = compute_metrics(csv_path,
                                 seeds=SEEDS_FILTER,
                                 distances=DISTANCES_FILTER)

    out_path = os.path.join(METRICS_DIR, f"metrics_{sc_name}.csv")
    df_metrics.to_csv(out_path, index=False)
    metrics_paths[sc_name] = out_path
    print(f"OK  →  {out_path}")

print(f"\n✓ Métricas guardadas en '{METRICS_DIR}'")


CSV encontrados: 12

  Procesando: scenario_01_ideal … OK  →  output_metrics\metrics_scenario_01_ideal.csv
  Procesando: scenario_02_canal_con_ruido … OK  →  output_metrics\metrics_scenario_02_canal_con_ruido.csv
  Procesando: scenario_03_canal_con_ruido_alto … OK  →  output_metrics\metrics_scenario_03_canal_con_ruido_alto.csv
  Procesando: scenario_04_ataque_eve_parcial … OK  →  output_metrics\metrics_scenario_04_ataque_eve_parcial.csv
  Procesando: scenario_05_ataque_eve_total … OK  →  output_metrics\metrics_scenario_05_ataque_eve_total.csv
  Procesando: scenario_06_fibra_deteriorada … OK  →  output_metrics\metrics_scenario_06_fibra_deteriorada.csv
  Procesando: scenario_07_detector_ruidoso … OK  →  output_metrics\metrics_scenario_07_detector_ruidoso.csv
  Procesando: scenario_08_baja_eficiencia_detector … OK  →  output_metrics\metrics_scenario_08_baja_eficiencia_detector.csv
  Procesando: scenario_09_baja_intensidad_fuente … OK  →  output_metrics\metrics_scenario_09_baja_intensidad_

## 5. Vista previa de métricas del escenario 1 (Ideal)

In [7]:
from pathlib import Path

ideal_key = [k for k in metrics_paths if "01" in k][0]
df_preview = pd.read_csv(metrics_paths[ideal_key])
cols_display = ["distancia","qber_mean","qber_std",
                "absorption_mean","absorption_std","skr_mean","skr_std"]
pd.set_option("display.float_format", "{:.4g}".format)
display(df_preview[cols_display])


,distancia,qber_mean,qber_std,absorption_mean,absorption_std,skr_mean,skr_std
0,0,0,0,0.1841,0.001048,0.1841,0.001048
1,5,0,0,0.1819,0.001184,0.1819,0.001183
2,10,0,0,0.1741,0.001081,0.1741,0.001081
3,15,0,0,0.1598,0.0008912,0.1598,0.0008962
4,20,0,0,0.1417,0.0007985,0.1417,0.0008053
5,25,0,0,0.1224,0.001007,0.1224,0.001008
6,30,0,0,0.1035,0.0008436,0.1035,0.0008444
7,35,0,0,0.08639,0.0006509,0.08639,0.0006492
8,40,0,0,0.071,0.000807,0.07099,0.0008057
9,45,0,0,0.0579,0.0006307,0.05789,0.0006275
